In [0]:
from bs4 import BeautifulSoup
from functools import reduce
from pyspark.sql import Row
import pyspark.sql.functions as f
import pyspark.pipelines as dp
import requests
import pyspark

In [0]:
venue_initials = {
    "T4B": "4Bs Tavern",
    "8BT": "8-Bit Arcade Bar",
    "AAB": "Add-a-Ball",
    "ADM": "Admiral Pub",
    "ANC": "Another Castle",
    "BSS": "Ballard Smoke Shop",
    "COP": "Corner Pocket Billiards and Lounge",
    "GRY": "Gary's Place",
    "GPA": "Georgetown Pizza and Arcade",
    "HND": "Hounds Tooth",
    "IBX": "Ice Box",
    "JUP": "Jupiter",
    "KRA": "Kraken",
    "OLF": "Olaf's",
    "RSW": "Rickshaw",
    "STN": "Seattle Tavern and Pool Hall",
    "SHR": "Shorty's",
    "TLT": "Tilted Table",
    "TWP": "Time Warp",
    "TDN": "Touchdown",
    "WAT": "Waterland",
    "ZTE": "Zoo Tavern",
}

In [0]:
venue_inits = list(venue_initials.keys())

In [0]:
def html_extract(url: str):

    html_src = requests.get(url).content
    soup = BeautifulSoup(html_src, "html.parser")
    tables = soup.find_all('table')

    # Extract data from HTML
    for table in tables:
        rows = table.find_all('tr')

        # First row contains the headers
        header_cells = rows[0].find_all(['th', 'td'])
        headers = [cell.text.strip() or f"column_{i}" for i, cell in enumerate(header_cells)]

        # Remaining rows are data
        table_data = []
        for row in rows[1:]:
            cells = row.find_all('td')
            row_dict = {headers[i]: cell.text.strip() for i, cell in enumerate(cells) if i < len(headers)}
            table_data.append(row_dict)

    return table_data

In [0]:
def create_df(venue_initial: str):

    data = html_extract(f"https://mondaynightpinball.com/venues/{venue_init}")

    df = spark.createDataFrame(data)
    header_row = spark.createDataFrame([Row(*df.columns)], df.schema)
    df = header_row.unionByName(df)
    df = df.withColumnRenamed(df.columns[0], "machine_name")
    df = df.withColumn("venue_init", f.lit(venue_initial))

    return df

In [0]:
df = create_df("TBT")

display(df)

In [0]:
dfs = []

for venue_init in venue_inits:
    df = create_df(venue_init)
    dfs.append(df)

df = reduce(pyspark.sql.dataframe.DataFrame.unionByName, dfs)


In [0]:
display(df)

In [0]:
venue_init = "T4B"
data = html_extract(f"https://mondaynightpinball.com/venues/{venue_init}")



df = spark.createDataFrame(data)


In [0]:
display(df)

In [0]:
header_row = spark.createDataFrame([Row(*df.columns)], df.schema)
df = header_row.unionByName(df)

In [0]:
df = df.withColumnRenamed("Cactus Canyon", "machine_name")
df = df.withColumn("venue_init", f.lit(venue_init))

In [0]:
display(df)